# Galaxy Zoo 2 - Exploratory Data Analysis

This notebook performs EDA on Galaxy Zoo 2 data for morphology classification.

## Contents
1. Data Loading & Overview
2. Label Distribution Analysis
3. Image Visualization
4. Morphological Parameter Analysis (CAS, Gini-M20)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

# Set style
plt.style.use('default')
sns.set_palette('husl')
%matplotlib inline

## 1. Data Loading

In [ ]:
# Paths - update these to your data location
DATA_DIR = Path('../data/raw')
LABELS_FILE = DATA_DIR / 'training_solutions_rev1.csv'  # Galaxy Zoo 2 labels
IMAGES_DIR = DATA_DIR / 'images_training_rev1'

# Load labels
df = pd.read_csv(LABELS_FILE)
print(f'Loaded {len(df):,} galaxy samples')
df.head()

## 2. Label Analysis

Galaxy Zoo 2 uses vote fractions for each question. For morphology:
- **Class1.1**: Smooth (elliptical-like)
- **Class1.2**: Features or disk (spiral-like)
- **Class1.3**: Star or artifact

In [ ]:
# Distribution of main morphology classes
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(['Class1.1', 'Class1.2', 'Class1.3']):
    axes[i].hist(df[col], bins=50, edgecolor='black', alpha=0.7)
    axes[i].set_xlabel(f'{col} Vote Fraction')
    axes[i].set_ylabel('Count')
    axes[i].set_title(f'{col} Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Binary classification with 80% threshold
THRESHOLD = 0.8

df['is_smooth'] = df['Class1.1'] >= THRESHOLD
df['is_featured'] = df['Class1.2'] >= THRESHOLD
df['confident'] = df['is_smooth'] | df['is_featured']

print(f"Confident classifications (>={THRESHOLD*100:.0f}% agreement):")
print(f"  Smooth (Elliptical): {df['is_smooth'].sum():,} ({df['is_smooth'].mean()*100:.1f}%)")
print(f"  Featured (Spiral):   {df['is_featured'].sum():,} ({df['is_featured'].mean()*100:.1f}%)")
print(f"  Total confident:     {df['confident'].sum():,} ({df['confident'].mean()*100:.1f}%)")

In [ ]:
# Vote entropy for label uncertainty
def entropy(row):
    probs = [row['Class1.1'], row['Class1.2'], row['Class1.3']]
    probs = [p for p in probs if p > 0]
    return -sum(p * np.log2(p) for p in probs)

df['vote_entropy'] = df.apply(entropy, axis=1)

plt.figure(figsize=(10, 4))
plt.hist(df['vote_entropy'], bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Vote Entropy (bits)')
plt.ylabel('Count')
plt.title('Label Uncertainty Distribution')
plt.axvline(x=1.0, color='r', linestyle='--', label='High uncertainty threshold')
plt.legend()
plt.show()

## 3. Image Visualization

In [ ]:
def load_galaxy_image(galaxy_id, images_dir=IMAGES_DIR):
    """Load galaxy image by ID."""
    img_path = images_dir / f'{galaxy_id}.jpg'
    if img_path.exists():
        return Image.open(img_path)
    return None

def show_sample_galaxies(df_subset, n=8, title='Sample Galaxies'):
    """Display a grid of galaxy images."""
    fig, axes = plt.subplots(2, n//2, figsize=(3*n//2, 6))
    axes = axes.flatten()
    
    for i, (_, row) in enumerate(df_subset.head(n).iterrows()):
        img = load_galaxy_image(row['GalaxyID'])
        if img:
            axes[i].imshow(img)
            axes[i].set_title(f"ID: {row['GalaxyID']}\nSmooth: {row['Class1.1']:.2f}")
        axes[i].axis('off')
    
    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# Show sample elliptical galaxies
ellipticals = df[df['is_smooth']].sample(8, random_state=42)
show_sample_galaxies(ellipticals, title='Sample Elliptical Galaxies (Smooth ≥ 80%)')

In [ ]:
# Show sample spiral galaxies
spirals = df[df['is_featured']].sample(8, random_state=42)
show_sample_galaxies(spirals, title='Sample Spiral Galaxies (Featured ≥ 80%)')

## 4. Morphological Parameters

Compute CAS and Gini-M20 for sample galaxies.

In [ ]:
from src.models.baseline import compute_morphological_params

# Sample subset for morphological analysis
sample_df = df[df['confident']].sample(500, random_state=42).copy()

# Compute morphological parameters
morph_params = []
for _, row in sample_df.iterrows():
    img = load_galaxy_image(row['GalaxyID'])
    if img:
        img_array = np.array(img)
        params = compute_morphological_params(img_array)
        params['GalaxyID'] = row['GalaxyID']
        morph_params.append(params)

morph_df = pd.DataFrame(morph_params)
sample_df = sample_df.merge(morph_df, on='GalaxyID')
print(f"Computed morphological parameters for {len(morph_df)} galaxies")

In [ ]:
# CAS vs morphology
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, param in enumerate(['concentration', 'asymmetry', 'smoothness']):
    for is_spiral, label, color in [(False, 'Elliptical', 'blue'), (True, 'Spiral', 'orange')]:
        subset = sample_df[sample_df['is_featured'] == is_spiral]
        axes[i].hist(subset[param], bins=30, alpha=0.5, label=label, color=color)
    axes[i].set_xlabel(param.capitalize())
    axes[i].set_ylabel('Count')
    axes[i].legend()

plt.suptitle('CAS Parameters by Morphology', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Gini-M20 diagram
plt.figure(figsize=(8, 6))

elliptical = sample_df[~sample_df['is_featured']]
spiral = sample_df[sample_df['is_featured']]

plt.scatter(elliptical['m20'], elliptical['gini'], alpha=0.5, label='Elliptical', c='blue')
plt.scatter(spiral['m20'], spiral['gini'], alpha=0.5, label='Spiral', c='orange')

plt.xlabel('M20')
plt.ylabel('Gini')
plt.title('Gini-M20 Diagram')
plt.legend()
plt.show()

## Summary

Key findings from EDA:
1. **Class Distribution**: Analyze imbalance between ellipticals and spirals
2. **Label Confidence**: Vote entropy shows classification uncertainty
3. **Morphological Parameters**: CAS and Gini-M20 show class separation